#  CalRetail — Personalised Recommendations
## Goal
Recommend top-N products most likely to be purchased by a customer using collaborative filtering and category preference boosting.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data' / 'processed'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from backend.utils.adaptive_thresholds import get_category_conversion_rates

# Load data
tx       = pd.read_csv(processed_dir / 'transactions.csv')
prod     = pd.read_csv(processed_dir / 'products.csv')
cust     = pd.read_csv(processed_dir / 'customers.csv')
cart     = pd.read_csv(processed_dir / 'shopping_cart.csv')
wishlist = pd.read_csv(processed_dir / 'wishlist.csv')

# ── Implicit-feedback matrix ─────────────────────────────────────────────────
# Purchases alone are sparse (most customers buy far less often than they
# browse/cart/wishlist). Blending all three REAL behavioural signals with
# decreasing weights gives denser, more accurate similarity vectors than
# purchase-only collaborative filtering.
PURCHASE_WEIGHT, CART_WEIGHT, WISHLIST_WEIGHT = 3.0, 2.0, 1.0

purchase_signal = tx.groupby(['customer_id', 'product_id'])['quantity'].sum() * PURCHASE_WEIGHT
cart_signal = (
    cart[cart['status'].isin(['Active', 'Abandoned'])]
    .groupby(['customer_id', 'product_id']).size() * CART_WEIGHT
)
wishlist_signal = wishlist.groupby(['customer_id', 'product_id']).size() * WISHLIST_WEIGHT

signal = purchase_signal.add(cart_signal, fill_value=0).add(wishlist_signal, fill_value=0)
matrix = signal.unstack(fill_value=0)

cust_sim = cosine_similarity(matrix)
cust_sim_df = pd.DataFrame(cust_sim, index=matrix.index, columns=matrix.index)

# Category -> conversion-boost multiplier, learned from real purchase/browse
# ratios (replaces a fixed 1.25x guess with each category's *actual* relative
# conversion strength in the live dataset).
category_boost = get_category_conversion_rates()

print(f"Implicit-feedback matrix computed: {matrix.shape} (purchases + cart + wishlist).")
print(f"Category boost multipliers (data-derived): {category_boost}")

In [ ]:
def _bestseller_fallback(pref_cat, top_n):
    """Cold-start / no-signal fallback: real bestsellers, preferred category first."""
    bestsellers = tx.groupby('product_id')['quantity'].sum()
    pool = prod[prod['product_id'].isin(bestsellers.index)].copy()
    pool['rank_score'] = pool['product_id'].map(bestsellers)
    if pref_cat:
        pool = pd.concat([pool[pool['category'] == pref_cat], pool[pool['category'] != pref_cat]])
    pool = pool.drop_duplicates('product_id').head(top_n * 3).nlargest(top_n, 'rank_score')
    return [{
        'product_id': r['product_id'], 'product_name': r['product_name'],
        'category': r['category'], 'price': float(r['price']),
        'score': round(float(r['rank_score']), 4),
        'reason': f"Popular in {pref_cat}" if pref_cat and r['category'] == pref_cat else "Trending bestseller"
    } for _, r in pool.iterrows()]


def get_recommendations(cust_id, top_n=5):
    pref_cat = None
    if cust_id in cust['customer_id'].values:
        pref_cat = cust.loc[cust['customer_id'] == cust_id, 'preferred_category'].iloc[0]

    if cust_id not in matrix.index:
        # True cold start: no purchase/cart/wishlist history yet.
        return {"customer_id": cust_id, "recommendations": _bestseller_fallback(pref_cat, top_n)}

    # Similarity-weighted collaborative filtering: neighbours contribute signal
    # proportional to how similar they are (standard user-based CF formula),
    # instead of treating the nearest 20 neighbours as equally important.
    sims = cust_sim_df[cust_id].drop(cust_id).nlargest(20)
    sims = sims[sims > 0]
    if len(sims) == 0:
        return {"customer_id": cust_id, "recommendations": _bestseller_fallback(pref_cat, top_n)}

    candidates = matrix.loc[sims.index].mul(sims.values, axis=0).sum(axis=0) / sims.sum()
    already_owned = matrix.columns[matrix.loc[cust_id] > 0]
    candidates = candidates.drop(index=already_owned, errors='ignore')

    results = []
    for pid, score in candidates.nlargest(top_n * 3).items():
        if score <= 0:
            continue
        p_info = prod[prod['product_id'] == pid].iloc[0]
        boost = category_boost.get(p_info['category'], 1.0)
        final_score = score * boost
        reason = (f"Top choice matching your preferred style: {pref_cat}"
                  if pref_cat and p_info['category'] == pref_cat
                  else f"Trending among similar shoppers ({p_info['category']} converts {boost:.2f}x avg)")
        results.append({
            'product_id': pid,
            'product_name': p_info['product_name'],
            'category': p_info['category'],
            'price': float(p_info['price']),
            'score': round(float(final_score), 4),
            'reason': reason
        })

    results = sorted(results, key=lambda x: x['score'], reverse=True)[:top_n]
    if not results:
        results = _bestseller_fallback(pref_cat, top_n)
    return {"customer_id": cust_id, "recommendations": results}

sample_cid = cust.iloc[0]['customer_id']
backend_res = get_recommendations(sample_cid, top_n=3)
print("Backend Recommendations API Output:\n", json.dumps(backend_res, indent=2))

In [ ]:
print("=== CALRETAIL RECOMMENDATIONS PANEL ===")
sample_cust = cust[cust['customer_id'] == sample_cid].iloc[0]
print(f"Active Customer: {sample_cust['name']} | Loyalty: {sample_cust['loyalty_tier']} | Preferred Segment: {sample_cust['preferred_category']}\n")
rec_df = pd.DataFrame(backend_res['recommendations'])
print(rec_df[['product_name', 'category', 'price', 'score', 'reason']].to_string(index=False))
